### Cell 1 — Import Libraries

In [1]:
import pandas as pd
import numpy as np
import warnings
import os
import pickle
warnings.filterwarnings('ignore')

os.chdir('/Users/cirrus/Desktop/PMOS_PROJECT/notebook')
pd.set_option('display.float_format', '{:.3f}'.format)
np.random.seed(42)

print('✅ Libraries imported')

✅ Libraries imported


### Cell 2 — Load Data & Risk Scores

In [3]:
# Load processed data
df = pd.read_csv('../data/processed/pmos_eda_clean.csv')
df.columns = df.columns.str.strip()

# Load risk scores
risk_scores = pd.read_csv('artifacts/patient_risk_scores.csv')

# Load models
with open('artifacts/best_model.pkl', 'rb') as f:
    best_model = pickle.load(f)

with open('artifacts/final_features.pkl', 'rb') as f:
    final_features = pickle.load(f)

with open('artifacts/risk_features.pkl', 'rb') as f:
    risk_features = pickle.load(f)

print('✅ Data and models loaded')
print(f'   Patients       : {len(df)}')
print(f'   Risk scores    : {risk_scores.shape}')
print(f'   Risk columns   : {list(risk_scores.columns)}')

✅ Data and models loaded
   Patients       : 541
   Risk scores    : (541, 9)
   Risk columns   : ['PCOS (Y/N)', 'Metabolic_Risk_prob', 'CVD_Risk_prob', 'Reproductive_Risk_prob', 'Psych_Risk_prob', 'Metabolic_Risk_label', 'CVD_Risk_label', 'Reproductive_Risk_label', 'Psych_Risk_label']


### Cell 3 — Define Recommendation Engine

In [9]:
def get_recommendations(metabolic_label, cvd_label,
                        repro_label, psych_label,
                        metabolic_prob, cvd_prob,
                        repro_prob, psych_prob):

    recommendations = []
    urgency_scores  = []

    # ── METABOLIC RISK ──
    if metabolic_label == 'High':
        urgency_scores.append(3)
        recommendations.append({
            'domain'  : '🔴 Metabolic Risk — Take Action',
            'subtitle': 'What you can do:',
            'actions' : [
                '🩺 Book a consultation — discuss your metabolic risk with an endocrinologist or physician',
                '🧪 Ask about — fasting glucose, HbA1c and insulin resistance tests your doctor considers appropriate',
                '🥗 Diet — prioritise protein, vegetables, whole grains and high-fibre foods; limit sugary drinks and refined carbohydrates',
                '🏃‍♀️ Activity — aim for around 150 minutes/week of moderate physical activity, if medically appropriate',
                '⚖️  Weight — if overweight, discuss a sustainable weight-management plan with your healthcare professional'
            ]
        })
    elif metabolic_label == 'Moderate':
        urgency_scores.append(2)
        recommendations.append({
            'domain'  : '🟡 Metabolic Risk — Focus on Prevention',
            'subtitle': 'What you can do:',
            'actions' : [
                '🩺 Schedule — annual metabolic screening with your doctor',
                '🥗 Diet — increase whole foods, reduce processed snacks and sugary drinks',
                '🏃‍♀️ Activity — 30 minutes of walking daily is a great starting point',
                '⚖️  Weight — monitor monthly, aim to maintain a healthy range'
            ]
        })
    else:
        urgency_scores.append(1)
        recommendations.append({
            'domain'  : '🟢 Metabolic Risk — Maintain & Monitor',
            'subtitle': 'What you can do:',
            'actions' : [
                '✅ Keep up your current healthy habits',
                '🧪 Annual metabolic check as routine preventive care'
            ]
        })

    # ── CVD RISK ──
    if cvd_label == 'High':
        urgency_scores.append(3)
        recommendations.append({
            'domain'  : '🔴 Cardiovascular Risk — Take Action',
            'subtitle': 'What you can do:',
            'actions' : [
                '🩺 Book a consultation — discuss your cardiovascular risk with a physician',
                '🧪 Ask about — blood pressure and a lipid profile (LDL, HDL, triglycerides)',
                '🥗 Diet — reduce excess saturated and trans fats and highly processed foods',
                '🏃‍♀️ Activity — maintain regular aerobic and strength activity as appropriate',
                '🚭 Smoking — avoid or quit smoking if applicable',
                '😴 Lifestyle — prioritise sleep and stress management'
            ]
        })
    elif cvd_label == 'Moderate':
        urgency_scores.append(2)
        recommendations.append({
            'domain'  : '🟡 Cardiovascular Risk — Focus on Prevention',
            'subtitle': 'What you can do:',
            'actions' : [
                '🧪 Annual lipid panel and blood pressure check',
                '🥗 Increase omega-3 rich foods — fish, walnuts, flaxseeds',
                '🏃‍♀️ Regular aerobic exercise for heart health',
                '😴 Aim for 7-8 hours of quality sleep'
            ]
        })
    else:
        urgency_scores.append(1)
        recommendations.append({
            'domain'  : '🟢 Cardiovascular Risk — Maintain & Monitor',
            'subtitle': 'What you can do:',
            'actions' : [
                '✅ Continue heart-healthy diet and activity habits',
                '🧪 Routine annual blood pressure check'
            ]
        })

    # ── REPRODUCTIVE RISK ──
    if repro_label == 'High':
        urgency_scores.append(3)
        recommendations.append({
            'domain'  : '🔴 Reproductive Risk — Take Action',
            'subtitle': 'What you can do:',
            'actions' : [
                '🩺 Book a consultation — speak with a gynaecologist or reproductive specialist',
                '🧪 Ask about — AMH levels, ovulation assessment and cycle regulation options',
                '📅 Track — monitor your menstrual cycle using an app or basal body temperature',
                '🌿 Lifestyle — stress reduction and healthy weight support hormonal balance',
                '💬 Family planning — if pregnancy is a goal, discuss timing and options early'
            ]
        })
    elif repro_label == 'Moderate':
        urgency_scores.append(2)
        recommendations.append({
            'domain'  : '🟡 Reproductive Risk — Focus on Prevention',
            'subtitle': 'What you can do:',
            'actions' : [
                '📅 Track your cycle monthly — note any irregularities',
                '🧪 Annual AMH test to monitor ovarian reserve',
                '🩺 Consult gynaecologist if planning pregnancy in the next 1-2 years'
            ]
        })
    else:
        urgency_scores.append(1)
        recommendations.append({
            'domain'  : '🟢 Reproductive Risk — Maintain & Monitor',
            'subtitle': 'What you can do:',
            'actions' : [
                '✅ Continue routine gynaecological care',
                '📅 Annual cycle tracking and checkup'
            ]
        })

    # ── PSYCHOLOGICAL RISK ──
    if psych_label == 'High':
        urgency_scores.append(3)
        recommendations.append({
            'domain'  : '🔴 Psychological Risk — Take Action',
            'subtitle': 'What you can do:',
            'actions' : [
                '💬 Talk to someone — consider speaking with a counsellor, psychologist or trusted person',
                '🧠 Ask about — mental health screening; your doctor can guide next steps',
                '🤝 Community — connect with PMOS support groups; shared experiences help',
                '🧘 Mindfulness — breathing exercises, meditation or gentle yoga can reduce stress',
                '🏃‍♀️ Activity — regular physical activity is one of the strongest mood regulators'
            ]
        })
    elif psych_label == 'Moderate':
        urgency_scores.append(2)
        recommendations.append({
            'domain'  : '🟡 Psychological Risk — Focus on Prevention',
            'subtitle': 'What you can do:',
            'actions' : [
                '📔 Mood tracking — journaling or a mood app can highlight patterns',
                '🏃‍♀️ Regular movement for natural mood regulation',
                '💬 Talk openly with your doctor about how PMOS symptoms affect you emotionally'
            ]
        })
    else:
        urgency_scores.append(1)
        recommendations.append({
            'domain'  : '🟢 Psychological Risk — Maintain & Monitor',
            'subtitle': 'What you can do:',
            'actions' : [
                '✅ Maintain social connections and regular self-care',
                '🧘 Continue stress management practices that work for you'
            ]
        })

    # ── OVERALL STATUS ──
    max_urgency = max(urgency_scores)
    if max_urgency == 3:
        overall = '🔴 HIGH RISK — Take Action'
        overall_sub = 'Here are practical next steps commonly recommended for people with this risk profile.'
    elif max_urgency == 2:
        overall = '🟡 MODERATE RISK — Focus on Prevention'
        overall_sub = 'Small consistent changes now can significantly reduce your long-term risk.'
    else:
        overall = '🟢 LOW RISK — Maintain & Monitor'
        overall_sub = 'Keep up your healthy habits and continue routine check-ups.'

    return {
        'recommendations': recommendations,
        'overall_urgency': overall,
        'overall_sub'    : overall_sub,
        'urgency_score'  : max_urgency
    }

print('✅ Recommendation engine defined — user-friendly version')

✅ Recommendation engine defined — user-friendly version


### Cell 4 — Test the Engine on Sample Patients

In [12]:
def display_recommendations(patient_id, result):
    print(f'\n{"="*60}')
    print(f'  PATIENT {patient_id} — PMOS MANAGEMENT PLAN')
    print(f'{"="*60}')
    print(f'  {result["overall_urgency"]}')
    print(f'  {result["overall_sub"]}')
    print(f'{"="*60}')
    
    for rec in result['recommendations']:
        print(f'\n{rec["domain"]}')
        print(f'  {rec["subtitle"]}')
        for action in rec['actions']:
            print(f'    {action}')
    print(f'\n{"="*60}\n')

# Test on 3 patients
test_patients = [
    {'id': 'A', 'metabolic': 'High',     'cvd': 'High',
     'repro': 'High',     'psych': 'High',
     'm_prob': 0.85, 'c_prob': 0.78, 'r_prob': 0.82, 'p_prob': 0.91},
    
    {'id': 'B', 'metabolic': 'Moderate', 'cvd': 'Low',
     'repro': 'High',     'psych': 'Moderate',
     'm_prob': 0.52, 'c_prob': 0.18, 'r_prob': 0.74, 'p_prob': 0.48},
    
    {'id': 'C', 'metabolic': 'Low',      'cvd': 'Low',
     'repro': 'Low',      'psych': 'Low',
     'm_prob': 0.12, 'c_prob': 0.08, 'r_prob': 0.21, 'p_prob': 0.15},
]

for p in test_patients:
    result = get_recommendations(
        metabolic_label=p['metabolic'],
        cvd_label      =p['cvd'],
        repro_label    =p['repro'],
        psych_label    =p['psych'],
        metabolic_prob =p['m_prob'],
        cvd_prob       =p['c_prob'],
        repro_prob     =p['r_prob'],
        psych_prob     =p['p_prob']
    )
    display_recommendations(p['id'], result)


  PATIENT A — PMOS MANAGEMENT PLAN
  🔴 HIGH RISK — Take Action
  Here are practical next steps commonly recommended for people with this risk profile.

🔴 Metabolic Risk — Take Action
  What you can do:
    🩺 Book a consultation — discuss your metabolic risk with an endocrinologist or physician
    🧪 Ask about — fasting glucose, HbA1c and insulin resistance tests your doctor considers appropriate
    🥗 Diet — prioritise protein, vegetables, whole grains and high-fibre foods; limit sugary drinks and refined carbohydrates
    🏃‍♀️ Activity — aim for around 150 minutes/week of moderate physical activity, if medically appropriate
    ⚖️  Weight — if overweight, discuss a sustainable weight-management plan with your healthcare professional

🔴 Cardiovascular Risk — Take Action
  What you can do:
    🩺 Book a consultation — discuss your cardiovascular risk with a physician
    🧪 Ask about — blood pressure and a lipid profile (LDL, HDL, triglycerides)
    🥗 Diet — reduce excess saturated and t

### Cell 5 — Apply to Real Patients & Save

In [13]:
# Apply recommendation engine to all 541 real patients
print('Generating recommendations for all patients...')

all_recommendations = []

for idx, row in risk_scores.iterrows():
    result = get_recommendations(
        metabolic_label=row['Metabolic_Risk_label'],
        cvd_label      =row['CVD_Risk_label'],
        repro_label    =row['Reproductive_Risk_label'],
        psych_label    =row['Psych_Risk_label'],
        metabolic_prob =row['Metabolic_Risk_prob'],
        cvd_prob       =row['CVD_Risk_prob'],
        repro_prob     =row['Reproductive_Risk_prob'],
        psych_prob     =row['Psych_Risk_prob']
    )
    all_recommendations.append({
        'patient_idx'    : idx,
        'PCOS (Y/N)'     : row['PCOS (Y/N)'],
        'overall_urgency': result['overall_urgency'],
        'urgency_score'  : result['urgency_score'],
        'metabolic_label': row['Metabolic_Risk_label'],
        'cvd_label'      : row['CVD_Risk_label'],
        'repro_label'    : row['Reproductive_Risk_label'],
        'psych_label'    : row['Psych_Risk_label'],
    })

rec_df = pd.DataFrame(all_recommendations)

print('✅ Recommendations generated for all patients')
print(f'\n=== URGENCY DISTRIBUTION ===')
print(rec_df['urgency_score'].value_counts().sort_index().to_dict())
print(f'\n  Score 1 (Stable)  : {(rec_df["urgency_score"]==1).sum()} patients')
print(f'  Score 2 (Monitor) : {(rec_df["urgency_score"]==2).sum()} patients')
print(f'  Score 3 (Urgent)  : {(rec_df["urgency_score"]==3).sum()} patients')

# Save
rec_df.to_csv('artifacts/patient_recommendations.csv', index=False)
with open('artifacts/recommendation_engine.pkl', 'wb') as f:
    pickle.dump(get_recommendations, f)

print('\n✅ Recommendations saved to artifacts/')
print('\n' + '='*50)
print('         BLOCK 8 COMPLETE')
print('='*50)
print('  Recommendation engine : ✅ built')
print('  Evidence-based rules  : ✅ 4 clinical domains')
print('  Urgency classification: ✅ 3 levels')
print('  All 541 patients      : ✅ scored')
print('  Saved for dashboard   : ✅')
print('='*50)
print('\n➡️  Next: Block 9 — Streamlit Dashboard')

Generating recommendations for all patients...
✅ Recommendations generated for all patients

=== URGENCY DISTRIBUTION ===
{1: 91, 2: 145, 3: 305}

  Score 1 (Stable)  : 91 patients
  Score 2 (Monitor) : 145 patients
  Score 3 (Urgent)  : 305 patients

✅ Recommendations saved to artifacts/

         BLOCK 8 COMPLETE
  Recommendation engine : ✅ built
  Evidence-based rules  : ✅ 4 clinical domains
  Urgency classification: ✅ 3 levels
  All 541 patients      : ✅ scored
  Saved for dashboard   : ✅

➡️  Next: Block 9 — Streamlit Dashboard
